

## load required pacakages

In [26]:
from evcouplings.align import Alignment
import os
import numpy as np
from pathlib import Path
import json
import pandas as pd

In [17]:
# --------------------------------------------------------------------
# Define updated gene coordinates with operon information
# --------------------------------------------------------------------
# Each gene maps to: [start_coord, end_coord, strand, is_operon]
updated_gene_coords = {
    "group_1": {
        "rrs-rrl":      [1471846, 1476795, "pos", False],
        "Rv0678":       [778990, 779487, "pos", False],
        "tlyA":         [1917940, 1918746, "pos", False],
        "embCAB":       [4239763, 4249810, "pos", True],
        "embC":         [4239863, 4243147, "pos", False],
        "embA":         [4243233, 4246517, "pos", False],
        "embB":         [4246513, 4249809, "pos", False],
        "ethAR":        [4326004, 4328199, "neg", True],
        "ethA":         [4326004, 4327473, "neg", False],
        "ethR":         [4327549, 4328199, "pos", False],
        "fabG1-inhA":   [1673440, 1675010, "pos", True],
        "fabG1":        [1673440, 1674183, "pos", False],
        "inhA":         [1674201, 1675010, "pos", False],
        "katG":         [2153889, 2156111, "neg", False],
        "eis":          [2714024, 2715432, "neg", False],
        "gyrBA":        [5140, 9918, "pos", True],
        "gyrB":         [5240, 7267, "pos", False],
        "gyrA":         [7302, 9818, "pos", False],
        "rplC":         [800809, 801462, "pos", False],
        "pncA":         [2288681, 2289241, "neg", False],
        "rpoBC":        [759807, 767325, "pos", False],
        "gid":          [4407528, 4408202, "neg", False],
        "rpsL":         [781560, 781934, "pos", False],
    }
}

# Save to JSON for external use or reproducibility
Path("updated_gene_coords.json").write_text(json.dumps(updated_gene_coords, indent=4))



2494

In [18]:
# --------------------------------------------------------------------
# Utility: Map alignment positions to genome coordinates for H37Rv
# --------------------------------------------------------------------
def build_genomic_coord_mapping(alignment, ref_id, genome_start_pos):
    """Create a mapping from MSA columns to genome coordinates based on reference sequence."""
    ref_seq = alignment.matrix[alignment.id_to_index[ref_id]]
    coord_mapping = []
    genome_pos = genome_start_pos

    for base in ref_seq:
        if base != "-":
            coord_mapping.append(genome_pos)
            genome_pos += 1
        else:
            coord_mapping.append(None)

    return coord_mapping

# --------------------------------------------------------------------
# Utility: Extract subset of columns from alignment corresponding to a genomic interval
# --------------------------------------------------------------------
def extract_gene_alignment(alignment, coord_mapping, gene_start, gene_end):
    """Select alignment columns corresponding to the gene's genomic region."""
    selected_cols = [
        idx for idx, pos in enumerate(coord_mapping)
        if pos is not None and gene_start <= pos < gene_end
    ]

    if not selected_cols:
        print(f"No columns found for genomic range {gene_start}–{gene_end}")
        return None, []

    subset_alignment = alignment.select(columns=selected_cols)
    selected_coords = [coord_mapping[i] for i in selected_cols]

    return subset_alignment, selected_coords



In [21]:
# --------------------------------------------------------------------
# File overrides for genes that share an operon or have combined FASTA files
# --------------------------------------------------------------------
gene_to_fasta_override = {
    "gyrA": "gyrBA.fasta", "gyrB": "gyrBA.fasta", "gyrBA": "gyrBA.fasta",
    "embC": "embCAB.fasta", "embA": "embCAB.fasta", "embB": "embCAB.fasta", "embCAB": "embCAB.fasta",
    "fabG1": "inhA.fasta", "inhA": "inhA.fasta", "fabG1-inhA": "inhA.fasta",
    "ethA": "ethA.fasta", "ethR": "ethR.fasta"
}

# --------------------------------------------------------------------
# Load alignment and build coordinate mappings for all genes
# --------------------------------------------------------------------
fasta_dir = "../data/combined_fasta_files"
gene_names = list(updated_gene_coords["group_1"].keys())

coord_columns = []
gene_to_file = {}

In [22]:
for gene_name in gene_names:
    start, _, strand, is_operon = updated_gene_coords["group_1"][gene_name]

    # Prefer explicit override mapping if available
    fasta_filename = gene_to_fasta_override.get(gene_name)
    if not fasta_filename:
        matching_files = [
            f for f in os.listdir(fasta_dir)
            if f.lower().endswith(".fasta") and gene_name.lower() in f.lower()
        ]
        if not matching_files:
            print(f"No FASTA file found for {gene_name}")
            continue
        fasta_filename = matching_files[0]

    file_path = os.path.join(fasta_dir, fasta_filename)
    try:
        alignment = Alignment.from_file(open(file_path), alphabet='-actg')
        ref_seq = alignment.matrix[alignment.id_to_index["MT_H37Rv"]]
    except Exception as e:
        print(f"Error loading alignment for {gene_name}: {e}")
        continue

    coord_mapping = build_genomic_coord_mapping(alignment, "MT_H37Rv", start)
    coord_columns.append(coord_mapping)
    gene_to_file[gene_name] = fasta_filename



No FASTA file found for ethAR


In [23]:
# --------------------------------------------------------------------
# Standardize: Pad all coordinate arrays to the same length for matrix creation
# --------------------------------------------------------------------
if not coord_columns:
    raise ValueError("No valid alignments were loaded. Verify FASTA availability and gene names.")

max_len = max(len(c) for c in coord_columns)
padded_coords = []

for c in coord_columns:
    padded = [pos if pos is not None else 0 for pos in c]
    padded += [0] * (max_len - len(padded))
    padded_coords.append(padded)

# --------------------------------------------------------------------
# Build final coordinate matrix (rows = MSA columns, columns = genes)
# --------------------------------------------------------------------
h37rv_matrix = np.array(padded_coords).T
print(f"Final matrix shape: {h37rv_matrix.shape} (rows: alignment columns, cols: genes)")
np.save("X_matrix_H37RV_coords_regenerated.npy", h37rv_matrix)


Final matrix shape: (10241, 22) (rows: alignment columns, cols: genes)


In [27]:
df_h37rv = pd.DataFrame(h37rv_numbers)
df_h37rv

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,1471846,778990,1917940,4239863,4239863,4243233,4246513,4326004,4327549,1673440,...,2153889,2714024,5140,5240,7302,800809,2288681,759807,4407528,781560
1,1471847,778991,1917941,4239864,4239864,4243234,4246514,4326005,4327550,1673441,...,2153890,2714025,5141,5241,7303,800810,2288682,759808,4407529,781561
2,1471848,778992,1917942,4239865,4239865,4243235,4246515,4326006,4327551,1673442,...,2153891,2714026,5142,5242,7304,800811,2288683,759809,4407530,781562
3,1471849,778993,1917943,4239866,4239866,4243236,4246516,4326007,4327552,1673443,...,2153892,2714027,5143,5243,7305,800812,2288684,759810,4407531,781563
4,1471850,778994,1917944,4239867,4239867,4243237,4246517,4326008,4327553,1673444,...,2153893,2714028,5144,5244,7306,800813,2288685,759811,4407532,781564
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10236,0,0,0,4250006,4250006,4253376,4256656,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10237,0,0,0,4250007,4250007,4253377,4256657,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10238,0,0,0,4250008,4250008,4253378,4256658,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10239,0,0,0,4250009,4250009,4253379,4256659,0,0,0,...,0,0,0,0,0,0,0,0,0,0
